Download all the requirements

In [ ]:
%pip install torch==2.7.1 torchvision torchaudio==2.7.1+cu126 --index-url https://download.pytorch.org/whl/cu126

In [ ]:
%pip install torchcodec ffmpeg-python pandas tqdm scikit-learn librosa birdnetlib birdnet resampy

In [ ]:
%pip freeze > requirements.txt

In [ ]:
# Download dataset
import kagglehub

# Download latest version
kagglehub.auth.set_kaggle_api_token('KGAT_fb4d6921c565524358c1914efc082ed4')
path = kagglehub.competition_download('birdclef-2026', output_dir=os.path.join("birdclef-2026"))

print("Path to competition files:", path)

Preprocessing the data

In [ ]:
# Helper - Get audio chunks with thresholding
from scipy import signal as scipy_signal
import numpy as np

# ──────────────────────────────────────────────────────────────────────────────
# Non-Aves: spectrogram peak detection via band-limited RMS
# ──────────────────────────────────────────────────────────────────────────────

def _get_spectrogram_active_chunks(
    audio_path:      str,
    chunk_sec:       float = 3.0,
    freq_min:        float = 100.0,
    freq_max:        float = 16_000.0,
    db_above_median: float = 6.0,
) -> list[tuple[float, float]]:
    """
    Detect active chunks via band-limited RMS energy.

    Reads the file once, band-pass filters to the species' vocal range, then
    slides a non-overlapping `chunk_sec` window.  A chunk is kept if its RMS
    (dBFS) is at least `db_above_median` dB above the per-file median.

    The adaptive threshold means each file self-calibrates to its own noise
    floor — no magic absolute dBFS value needed.

    Parameters
    ----------
    freq_min / freq_max
        Bandpass range in Hz.  Tune per species group:
          frogs/toads   →  100 –  4 000 Hz
          mammals       →   50 –  8 000 Hz  (bats: 10 000 – 120 000)
          insects       →  500 – 20 000 Hz
          broadband     →  100 – 16 000 Hz  ← default; safe starting point
    db_above_median
         3  → aggressive  (catches faint/distant calls, more false positives)
         6  → balanced    ← default
        10  → conservative (only prominent peaks)

    Returns
    -------
    List of (start_sec, end_sec) tuples for chunks that exceed the threshold.
    """
    audio, sr = sf.read(audio_path, dtype="float32", always_2d=False)
    if audio.ndim > 1:
        audio = audio.mean(axis=1)              # stereo → mono

    nyq          = sr / 2.0
    freq_max_clipped = min(freq_max, nyq * 0.95)   # stay below Nyquist

    sos      = scipy_signal.butter(
        4, [freq_min / nyq, freq_max_clipped / nyq], btype="band", output="sos"
    )
    audio_bp = scipy_signal.sosfilt(sos, audio)

    chunk_samples = int(chunk_sec * sr)
    duration      = len(audio) / sr

    energies: list[float]               = []
    bounds:   list[tuple[float, float]] = []

    for chunk_start in range(0, int(duration - chunk_sec + 1), int(chunk_sec)):
        s     = int(chunk_start * sr)
        chunk = audio_bp[s : s + chunk_samples]
        rms   = np.sqrt(np.mean(chunk ** 2))
        energies.append(20.0 * np.log10(rms + 1e-10))
        bounds.append((float(chunk_start), float(chunk_start + chunk_sec)))

    if not energies:
        return []

    threshold = float(np.median(energies)) + db_above_median
    return [(s, e) for (s, e), en in zip(bounds, energies) if en >= threshold]

In [ ]:
# Manifest generator
import soundfile as sf
from birdnetlib import Recording
from birdnetlib.analyzer import Analyzer
import logging
import contextlib
import io
import torchaudio
import pandas as pd

logging.getLogger("birdnetlib").setLevel(logging.ERROR)

# Initialise once — expensive to reload each call
analyzer = Analyzer()
silent = contextlib.redirect_stdout(io.StringIO())

BIRDNET_SR        = 48000
BIRDNET_CHUNK_SEC = 3
BIRDNET_MIN_CONF  = 0.35   # lower = permissive (catches quiet/distant birds)
                           # raise to 0.25+ if you're getting too much noise through

def generate_manifest(root_dir, output_csv="manifest.csv"):
    manifest_data = []
    skipped_no_bird = 0
    skipped_no_peak = 0

    full_df     = pd.read_csv(os.path.join(root_dir, "train.csv"))

    for _, entry in tqdm(full_df.iterrows(), total=len(full_df)):
        curr_audio_loc = os.path.join(root_dir, "train_audio", entry["filename"])
        label = entry['primary_label']

        try:
            if entry['class_name'] == 'Aves':
                # ONE BirdNET call for the whole file — no temp files, no manual chunking
                rec = Recording(analyzer, curr_audio_loc, min_conf=BIRDNET_MIN_CONF)
                with silent:
                    rec.analyze()

                # Build a lookup of which 3s windows had detections
                # BirdNET uses 1.5s steps internally so detections won't perfectly 
                # align to our chunk boundaries — check for overlap instead
                detections = rec.detections  # list of dicts with start_time, end_time

                # Get file duration without loading the whole waveform
                info     = sf.info(curr_audio_loc)
                duration = info.duration

                for chunk_start in range(0, int(duration - BIRDNET_CHUNK_SEC + 1), BIRDNET_CHUNK_SEC):
                    chunk_end = chunk_start + BIRDNET_CHUNK_SEC

                    if not _window_has_detection(detections, chunk_start, chunk_end):
                        skipped_no_bird += 1
                        continue

                    manifest_data.append({
                        'filename':      curr_audio_loc,
                        'start_sec':     float(chunk_start),
                        'end_sec':       float(chunk_end),
                        'class':         entry['class_name'],
                        'primary_label': label
                    })
            else:
                # ── Non-Aves: spectrogram energy peak detection ──────────────
                active_chunks = _get_spectrogram_active_chunks(curr_audio_loc)

                if not active_chunks:
                    skipped_no_peak += 1
                    continue

                for chunk_start, chunk_end in active_chunks:
                    manifest_data.append({
                        'filename':      curr_audio_loc,
                        'start_sec':     chunk_start,
                        'end_sec':       chunk_end,
                        'class':         entry['class_name'],
                        'primary_label': label,
                    })

        except Exception as e:
            print(f"Error processing {curr_audio_loc}: {e}")

    df = pd.DataFrame(manifest_data)
    df.to_csv(output_csv, index=False)
    print(f"Manifest created: {len(df)} chunks kept, {skipped_no_bird} rejected.")


def _window_has_detection(detections: list, chunk_start: float, chunk_end: float) -> bool:
    """
    True if any BirdNET detection overlaps with [chunk_start, chunk_end].
    BirdNET's windows are 3s with 1.5s steps, so a detection at t=1.5
    covers [1.5, 4.5] and overlaps both chunk 0-3 and chunk 3-6.
    """
    for det in detections:
        if det['start_time'] < chunk_end and det['end_time'] > chunk_start:
            return True
    return False

# generate_manifest(root_dir=os.path.join("..", "birdclef-2026"), output_csv=os.path.join("..", "birdclef-2026", "train_preproc_non_ave.csv"))

Dataset and CNN

In [ ]:
# Helper - Include one entry at least in each set
import warnings

def stratified_split(
    df,
    val_size   = 0.2,
    label_col  = 'primary_label',
    random_state = 42,
):
    """
    Train / val split that guarantees every class appears in both sets.

    Classes with exactly 1 sample cannot be split — they are placed in train
    and *also copied into val* so every label is present at evaluation time.
    A warning lists the affected classes so you can decide whether to collect
    more data for them.

    Parameters
    ----------
    df           : manifest DataFrame (output of generate_manifest)
    val_size     : fraction of splittable rows that go to val (default 0.2)
    label_col    : column containing the class label
    random_state : for reproducibility

    Returns
    -------
    df_train, df_val — DataFrames with reset indices
    """
    counts          = df[label_col].value_counts()
    singleton_mask  = df[label_col].isin(counts[counts == 1].index)
    singletons      = df[singleton_mask]
    splittable      = df[~singleton_mask]

    if not singletons.empty:
        affected = sorted(singletons[label_col].unique().tolist())
        warnings.warn(
            f"{len(affected)} class(es) have only 1 sample and will be duplicated "
            f"into both splits: {affected}",
            UserWarning,
            stacklevel=2,
        )

    if not splittable.empty:
        train_split, val_split = train_test_split(
            splittable,
            test_size    = val_size,
            stratify     = splittable[label_col],
            random_state = random_state,
        )
    else:
        train_split = pd.DataFrame(columns=df.columns)
        val_split   = pd.DataFrame(columns=df.columns)

    df_train = pd.concat([train_split, singletons], ignore_index=True)
    df_val   = pd.concat([val_split,   singletons], ignore_index=True)

    # Sanity check — should never fire
    missing_from_val = set(df_train[label_col]) - set(df_val[label_col])
    if missing_from_val:
        raise RuntimeError(f"BUG: these classes are absent from val: {missing_from_val}")

    return df_train, df_val

In [ ]:
# Dataset class
from torch.utils.data import Dataset

class BirbSet(Dataset):
    # Possibly add a download flag, for now assume it is on device
    # Add a Train flag to retrieve the sets accordingly
    def __init__(self, df, root, clip_length, label_to_idx, is_train = False, sample_rate= 32000):
        self.clips = []
        self.start_times = []
        self.end_times = []
        self.labels = []

        self.sample_rate = sample_rate
        self.clip_length = clip_length
        self.label_to_idx = label_to_idx
        self.is_train = is_train
        self.root = root

        self.amp_to_db = torchaudio.transforms.AmplitudeToDB(stype='power')
        self.mel_spect = torchaudio.transforms.MelSpectrogram(
            sample_rate=self.sample_rate,
            n_fft=800,
            n_mels=64
        )

        self.time_mask = torchaudio.transforms.TimeMasking(time_mask_param=40)
        self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=16)
        
        for i, entry in df.iterrows():
            curr_audio_loc = os.path.normpath(entry["filename"])
            self.clips.append(curr_audio_loc)
            self.labels.append(self.label_to_idx[entry['primary_label']])
            self.start_times.append(entry["start_sec"])
            self.end_times.append(entry["end_sec"])

    def __len__(self):
        return len(self.clips)
    
    def __getitem__(self, idx):
        audio_clip = self.clips[idx]
        label      = self.labels[idx]
        try:
            frame_offset = int(self.start_times[idx] * self.sample_rate)
            num_frames   = int((self.end_times[idx] - self.start_times[idx]) * self.sample_rate)

            waveform, sr = torchaudio.load(
                audio_clip, frame_offset=frame_offset, num_frames=num_frames
            )

            if sr != self.sample_rate:
                waveform = torchaudio.transforms.Resample(sr, self.sample_rate)(waveform)

            chunk_size  = self.sample_rate * self.clip_length
            current_len = waveform.shape[1]

            if current_len > chunk_size:
                waveform = waveform[:, :chunk_size]
            elif current_len < chunk_size:
                waveform = torch.nn.functional.pad(waveform, (0, chunk_size - current_len))

            spectrogram = self.mel_spect(waveform)
            spectrogram = self.amp_to_db(spectrogram)
            mean, std   = spectrogram.mean(), spectrogram.std() + 1e-6
            spectrogram = (spectrogram - mean) / std

            if self.is_train:
                spectrogram = self.freq_mask(spectrogram)
                spectrogram = self.time_mask(spectrogram)

            return spectrogram, label
        except Exception as e:
            print(f"Failed to load audio clip at index {idx} -> {e}.\nFile location {self.clips[idx]}")

In [ ]:
# CNN
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights

class EfficientBirbNN(nn.Module):
    def __init__(self, num_classes, pretrained=True):
        super().__init__()
        
        # 1. Load the base EfficientNet model
        weights = EfficientNet_B3_Weights.DEFAULT if pretrained else None
        self.base_model = efficientnet_b3(weights=weights)
        
        # 2. Modify the first convolutional layer to accept 1-channel spectrograms
        # EfficientNet's first layer is located at self.base_model.features[0][0]
        original_conv = self.base_model.features[0][0]
        self.base_model.features[0][0] = nn.Conv2d(
            in_channels=1, 
            out_channels=original_conv.out_channels, 
            kernel_size=original_conv.kernel_size, 
            stride=original_conv.stride, 
            padding=original_conv.padding, 
            bias=False
        )
        
        # (Optional but recommended) Initialize the new 1-channel conv 
        # by averaging the pre-trained 3-channel weights to retain feature extraction strength
        if pretrained:
            with torch.no_grad():
                self.base_model.features[0][0].weight[:] = original_conv.weight.mean(dim=1, keepdim=True)
                
        # 3. Modify the final classification layer for your specific number of bird classes
        in_features = self.base_model.classifier[1].in_features
        self.base_model.classifier[1] = nn.Sequential(
            nn.Dropout(p=0.4), # Extra dropout to prevent overfitting on audio data
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        return self.base_model(x)

In [11]:
# Generate data sets
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

root_path = os.path.join("..", "birdclef-2026")
BIRDNET_CHUNK_SEC = 3.0

# 1. Load and filter the master CSV
full_df = pd.read_csv(os.path.join(root_path, "train_preproc.csv"))

# 2. Split by unique files to prevent audio bleed between train/test
unique_files = full_df['filename'].unique().tolist()
train_files, test_files = train_test_split(unique_files, train_size=0.8, test_size=0.2, random_state=42)

train_df = full_df[full_df['filename'].isin(train_files)].reset_index(drop=True)
test_df = full_df[full_df['filename'].isin(test_files)].reset_index(drop=True)

# 3. Create a universal label mapping
unique_labels = pd.unique(full_df['primary_label'])
master_label_to_idx = {label: i for i, label in enumerate(unique_labels)}

num_classes  = len(master_label_to_idx)
class_counts = (
    train_df['primary_label']
    .map(master_label_to_idx)
    .value_counts()
    .reindex(range(num_classes), fill_value=1)  # fill_value=1 avoids div-by-zero
    .sort_index()
)
weights = 1.0 / torch.tensor(class_counts.values, dtype=torch.float)
weights = weights / weights.sum()

# 4. Initialize DataLoaders
dset = BirbSet(df=train_df, root=root_path, clip_length=BIRDNET_CHUNK_SEC, label_to_idx=master_label_to_idx)
loader = DataLoader(dset, batch_size=32, shuffle=True, pin_memory=True)

dset_test = BirbSet(df=test_df, root=root_path, clip_length=BIRDNET_CHUNK_SEC, label_to_idx=master_label_to_idx)
loader_test = DataLoader(dset_test, batch_size=32, shuffle=False, pin_memory=True)

Training and validation

In [ ]:
# Hyper params
device = torch.device("cuda")
model = EfficientBirbNN(num_classes=len(unique_labels)).to(device)
loss = nn.CrossEntropyLoss(weight=weights.to(device), label_smoothing=0.1)
optimiser = torch.optim.AdamW(model.parameters(), lr=0.0001, weight_decay=1e-4)

In [ ]:
# Train and validate
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import copy
from tqdm import tqdm

def validate(model, loader_test, loss_fn, master_label_to_idx, test_df, device, min_val_support=50):
    idx_to_label = {v: k for k, v in master_label_to_idx.items()}

    evaluable_classes = {
        label for label, count in
        test_df['primary_label'].value_counts().items()
        if count >= min_val_support
    }
    evaluable_idx = {master_label_to_idx[l] for l in evaluable_classes}
    print(f"Evaluating on {len(evaluable_classes)}/{len(master_label_to_idx)} classes")

    # ── Inference ─────────────────────────────────────────────────────────────
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    val_losses = []

    with torch.no_grad():
        for spects, labels in tqdm(loader_test, desc="Validating"):
            spects, labels = spects.to(device), labels.to(device)
            logits = model(spects)
            probs  = torch.softmax(logits, dim=1)

            val_losses.append(loss_fn(logits, labels).item())
            all_probs.append(probs.cpu())
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_probs  = torch.cat(all_probs)
    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)

    # ── Filter rare classes ────────────────────────────────────────────────────
    eval_mask  = np.array([l in evaluable_idx for l in all_labels])
    all_preds  = all_preds[eval_mask]
    all_labels = all_labels[eval_mask]
    all_probs  = all_probs[eval_mask]
    print(f"Filtered to {eval_mask.sum()} samples for evaluation")

    # ── Overview ──────────────────────────────────────────────────────────────
    avg_loss = sum(val_losses) / len(val_losses)
    top1     = (all_preds == all_labels).mean() * 100
    print(f"\nVal loss: {avg_loss:.4f} | Top-1 acc: {top1:.2f}%")

    # ── Top-K accuracy ─────────────────────────────────────────────────────────
    def topk_acc(probs, labels, k):
        topk = probs.topk(k, dim=1).indices
        return topk.eq(torch.tensor(labels).unsqueeze(1)).any(dim=1).float().mean().item() * 100

    print(f"\nTop-K accuracy:")
    print(f"  Top-1 : {topk_acc(all_probs, all_labels, 1):>6.2f}%")
    print(f"  Top-3 : {topk_acc(all_probs, all_labels, 3):>6.2f}%")
    print(f"  Top-5 : {topk_acc(all_probs, all_labels, 5):>6.2f}%")
    print(f"  Top-10: {topk_acc(all_probs, all_labels, 10):>6.2f}%")

    # ── Confidence split ───────────────────────────────────────────────────────
    correct_mask = all_preds == all_labels
    max_conf     = all_probs.max(dim=1).values.numpy()

    print(f"\nConfidence:")
    print(f"  Mean — correct:         {max_conf[correct_mask].mean():.3f}")
    print(f"  Mean — incorrect:       {max_conf[~correct_mask].mean():.3f}")
    print(f"  >90% confident & wrong: {(max_conf[~correct_mask] > 0.9).mean() * 100:.1f}%")

    # ── Per-class breakdown ────────────────────────────────────────────────────
    present_labels = sorted(set(all_labels))
    present_names  = [idx_to_label[i] for i in present_labels]

    report = classification_report(
        all_labels, all_preds,
        labels=present_labels,
        target_names=present_names,
        zero_division=0,
        output_dict=True
    )

    class_stats = [
        (name, report[name]['recall'], report[name]['f1-score'], int(report[name]['support']))
        for name in present_names
    ]

    header  = f"\n{'Class':<25} {'Recall':>7} {'F1':>7} {'Support':>9}"
    divider = "-" * 52

    print(f"\n--- 15 worst classes by recall ---{header}\n{divider}")
    for name, recall, f1, support in sorted(class_stats, key=lambda x: x[1])[:15]:
        print(f"{name:<25} {recall:>7.3f} {f1:>7.3f} {support:>9}")

    print(f"\n--- 15 best classes by recall ---{header}\n{divider}")
    for name, recall, f1, support in sorted(class_stats, key=lambda x: x[1], reverse=True)[:15]:
        print(f"{name:<25} {recall:>7.3f} {f1:>7.3f} {support:>9}")

    # ── Most confused pairs ────────────────────────────────────────────────────
    cm = confusion_matrix(all_labels, all_preds, labels=present_labels)
    np.fill_diagonal(cm, 0)

    confused = [
        (cm[i, j], present_names[i], present_names[j])
        for i in range(len(present_names))
        for j in range(len(present_names))
        if cm[i, j] > 0
    ]

    print(f"\n--- Top 20 confused pairs (true → predicted) ---")
    print(f"{'True class':<25} {'Predicted as':<25} {'Count':>7}")
    print("-" * 60)
    for count, true_cls, pred_cls in sorted(confused, reverse=True)[:20]:
        print(f"{true_cls:<25} {pred_cls:<25} {count:>7}")

    return avg_loss, top1


# ── Training loop ──────────────────────────────────────────────────────────────
epochs           = 30
best_val_acc     = 0.0
best_model_state = None
patience         = 8
epochs_no_improve = 0

loss_fn   = nn.CrossEntropyLoss(weight=weights.to(device))
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimiser, T_max=epochs)

for epoch in range(epochs):
    # ── Training ───────────────────────────────────────────────────────────────
    model.train()
    train_losses = []
    for spects, labels in tqdm(loader, desc=f"Epoch {epoch+1}/{epochs} [train]"):
        spects, labels = spects.to(device), labels.to(device)
        optimiser.zero_grad()
        l = loss(model(spects), labels)
        l.backward()
        optimiser.step()
        train_losses.append(l.item())

    scheduler.step()

    # ── Validation ─────────────────────────────────────────────────────────────
    avg_train = sum(train_losses) / len(train_losses)
    avg_val, val_acc = validate(
        model, loader_test, loss_fn, master_label_to_idx, test_df, device
    )

    print(f"Epoch {epoch+1:2d} | Train loss: {avg_train:.4f} | "
          f"Val loss: {avg_val:.4f} | Val acc: {val_acc:.2f}%")

    # ── Early stopping ─────────────────────────────────────────────────────────
    if val_acc > best_val_acc:
        best_val_acc     = val_acc
        best_model_state = copy.deepcopy(model.state_dict())
        epochs_no_improve = 0
        torch.save(best_model_state, "best_birb_model.pt")
        print(f"  → New best model saved ({best_val_acc:.2f}%)")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

model.load_state_dict(best_model_state)